# ATE / CATE benchmarks

In [ ]:
import numpy as np
from scipy import stats
from sklearn.linear_model import LogisticRegression, Ridge

from causal_medmnist import Scenario
from causal_medmnist.datasets import REGISTRY

In [ ]:
n_samples = 1000
num_repetitions = 200
propensity_clip = 0.03


def generate(scenario, n, rng, split):
    sample = scenario.generate(n, seed=int(rng.integers(0, 2**32)), split=split, replace=True)
    return sample.X, sample.A, sample.Y.reshape(n, -1).mean(axis=1)


def pseudo_outcomes(X_train, A_train, y_train, X_test, A_test, y_test):
    propensity_model = LogisticRegression().fit(X_train, A_train)
    mu0 = Ridge().fit(X_train[A_train == 0], y_train[A_train == 0])
    mu1 = Ridge().fit(X_train[A_train == 1], y_train[A_train == 1])

    pi = np.clip(propensity_model.predict_proba(X_test)[:, 1], propensity_clip, 1 - propensity_clip)
    m0, m1 = mu0.predict(X_test), mu1.predict(X_test)
    return m1 - m0 + A_test * (y_test - m1) / pi - (1 - A_test) * (y_test - m0) / (1 - pi)


def aipw_test(X, phi):
    z = np.sqrt(len(phi)) * phi.mean() / phi.std(ddof=1)
    return 2 * stats.norm.sf(abs(z))


def cate_test(X, phi):
    n, p = X.shape
    design = np.column_stack([np.ones(n), X])
    beta = np.linalg.lstsq(design, phi, rcond=None)[0]
    residual = phi - design @ beta
    bread = np.linalg.inv(design.T @ design)
    cov = bread @ (design.T * residual**2) @ design @ bread * (n / (n - p - 1))
    wald = beta[1:] @ np.linalg.solve(cov[1:, 1:], beta[1:])
    return stats.chi2.sf(wald, p)


def run_test(dataset, n_samples, scale, heterogeneity, test, rng):
    scenario = Scenario(dataset, effect_strength=scale, distributional_effect=False, effect_heterogeneity=heterogeneity)
    rejections = []

    for _ in range(num_repetitions):
        X_train, A_train, y_train = generate(scenario, n_samples // 2, rng, "train")
        X_test, A_test, y_test = generate(scenario, n_samples - n_samples // 2, rng, "val")

        phi = pseudo_outcomes(X_train, A_train, y_train, X_test, A_test, y_test)
        pvalues = test(X_test, phi)

        rejections.append(float(pvalues < 0.05))

    return np.mean(rejections)


def run(scale, heterogeneity, test, rng):
    for dataset in sorted(REGISTRY):
        result = run_test(dataset, n_samples, scale, heterogeneity, test, rng)
        print(f"[{dataset}][N={n_samples}][scale={scale}][{test.__name__}] {result}")

In [ ]:
rng = np.random.default_rng(0)

run(scale=0.6, heterogeneity=0.0, test=aipw_test, rng=rng)
run(scale=0.6, heterogeneity=2.0, test=cate_test, rng=rng)